In [28]:
import os
import glob
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [29]:
# --- 1. Define Paths ---
TEST_SEGMENT_DIR = "/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/test_segment/test_segment/"
TRAIN_DIR = "/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/train/train/"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/sample_submission.csv"

ROWS_PER_SEGMENT = 480

In [30]:
# --- 2. Advanced Feature Extraction ---
def extract_advanced_features(segment_df):
    features = {}
    sensor_cols = ['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'TEMP', 'EDA', 'HR', 'IBI']
    
    for col in sensor_cols:
        if col in segment_df.columns and not segment_df[col].empty:
            data = segment_df[col].values
            features[f'{col}_mean'] = np.mean(data)
            features[f'{col}_std'] = np.std(data)
            features[f'{col}_min'] = np.min(data)
            features[f'{col}_max'] = np.max(data)
            features[f'{col}_median'] = np.median(data)
            # Add percentiles (great for ignoring sudden sensor spikes)
            features[f'{col}_p25'] = np.percentile(data, 25)
            features[f'{col}_p75'] = np.percentile(data, 75)
            # Add signal range
            features[f'{col}_range'] = np.max(data) - np.min(data)
        else:
            # fill with zeros if missing
            for stat in ['mean', 'std', 'min', 'max', 'median', 'p25', 'p75', 'range']:
                features[f'{col}_{stat}'] = 0
                
    return features

In [31]:
# --- 3. Process Training Data ---
print("Processing Training Data...")
X_train_list = []
y_train_list = []

train_files = glob.glob(os.path.join(TRAIN_DIR, "*.csv"))

for file in tqdm(train_files, desc="Reading Train CSVs"):
    try:
        df = pd.read_csv(file)
        
        for i in range(0, len(df), ROWS_PER_SEGMENT):
            segment = df.iloc[i : i + ROWS_PER_SEGMENT]
            
            if len(segment) < ROWS_PER_SEGMENT:
                continue
                
            features = extract_advanced_features(segment)
            X_train_list.append(features)
            y_train_list.append(segment['Sleep_Stage'].mode()[0])
            
    except Exception as e:
        print(f"Error processing {file}: {e}")

X_train = pd.DataFrame(X_train_list)
y_train = np.array(y_train_list)

# Encode labels (XGBoost requires target labels to start from 0)
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

Processing Training Data...


Reading Train CSVs: 100%|██████████| 83/83 [04:28<00:00,  3.23s/it]


In [32]:
# --- 4. Process Test Data ---
print("Processing Test Data...")
sample_df = pd.read_csv(SAMPLE_SUB_PATH)
X_test_list = []

for test_id in tqdm(sample_df['id'], desc="Processing Test Segments"):
    folder_name = test_id.split('_')[0]
    
    file_path_1 = os.path.join(TEST_SEGMENT_DIR, folder_name, f"{test_id}.csv")
    file_path_2 = os.path.join(TEST_SEGMENT_DIR, folder_name, f"{test_id.split('_')[1]}.csv")
    
    target_path = file_path_1 if os.path.exists(file_path_1) else (file_path_2 if os.path.exists(file_path_2) else None)
        
    if target_path:
        segment_df = pd.read_csv(target_path)
        features = extract_advanced_features(segment_df)
    else:
        print(f"Warning: File for {test_id} not found. Filling with zeros.")
        features = extract_advanced_features(pd.DataFrame(columns=['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'TEMP', 'EDA', 'HR', 'IBI']))
        
    X_test_list.append(features)

X_test = pd.DataFrame(X_test_list)

Processing Test Data...


Processing Test Segments: 100%|██████████| 7832/7832 [00:50<00:00, 156.44it/s]


In [33]:
# --- 5. Train XGBoost Model ---
print("Training XGBoost Classifier...")
# These are solid starting hyperparameters for tabular biosignal data
xgb_model = XGBClassifier(
    n_estimators=300,        # Number of trees
    learning_rate=0.05,      # Step size shrinkage
    max_depth=6,             # Depth of the trees
    subsample=0.8,           # Use 80% of data per tree to prevent overfitting
    colsample_bytree=0.8,    # Use 80% of features per tree
    random_state=42,
    n_jobs=-1                # Use all CPU cores
)

xgb_model.fit(X_train, y_train_encoded)

Training XGBoost Classifier...


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=-1, num_parallel_tree=None, ...)

In [34]:
# --- 6. Predict and Generate Submission ---
print("Generating Predictions...")
preds_encoded = xgb_model.predict(X_test)
preds_labels = label_encoder.inverse_transform(preds_encoded)

sample_df['labels'] = preds_labels
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Make sure to save to /kaggle/working/ to avoid read-only errors
submission_filename = f"/kaggle/working/submission_xgboost_{timestamp}.csv"

sample_df.to_csv(submission_filename, index=False)
print(f"Pipeline complete! Submission saved to: {submission_filename}")

Generating Predictions...
Pipeline complete! Submission saved to: /kaggle/working/submission_xgboost_20260403_203341.csv
